# Clase 059 — Launch, monitoreo y mantenimiento de modelos

Entrenar es la mitad: el resto es serializar, detectar **data drift** (PSI, KS-test),
disparar retraining y comparar releases (shadow deploy). Simulamos todo de forma ejecutable.

Requiere: `numpy`, `pandas`, `scipy`, `scikit-learn`, `joblib`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import joblib, tempfile, os
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

np.random.seed(42)
rng = np.random.default_rng(42)

## 1. Serializar y recargar (artefacto productivo)

El modelo serializado con `joblib` es el artefacto que va a producción. Debe predecir
idéntico tras recargar.

In [ ]:
X, y = make_classification(n_samples=3000, n_features=8, n_informative=5, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
clf = RandomForestClassifier(n_estimators=120, random_state=42).fit(Xtr, ytr)
print(f'accuracy test: {clf.score(Xte, yte):.4f}')

path = os.path.join(tempfile.gettempdir(), 'modelo_prod.joblib')
joblib.dump(clf, path)
loaded = joblib.load(path)
assert np.array_equal(loaded.predict(Xte), clf.predict(Xte)), 'debe predecir identico'
print('OK: modelo recargado predice identico')

## 2. Simular dos snapshots: producción con drift inyectado

Mes 1 = distribución de entrenamiento. Mes 6 = una feature drifteada (media desplazada).
Así se ve el *covariate shift* en la práctica.

In [ ]:
feat = 0
snap1 = Xte.copy()
snap2 = Xte.copy()
snap2[:, feat] = snap2[:, feat] + 2.0   # drift: la feature 0 se corre +2 sigmas
print(f'feature {feat}: media mes1 {snap1[:, feat].mean():.3f} '
      f'-> media mes6 {snap2[:, feat].mean():.3f}')

## 3. Population Stability Index (PSI) por feature

PSI cuantifica el cambio de distribución. Umbrales de industria: <0.1 estable, 0.1-0.25 leve,
>0.25 drift significativo. Sumamos epsilon para evitar log(0).

In [ ]:
def psi(expected, actual, bins=10, eps=1e-6):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + eps
    a = np.histogram(actual, edges)[0] / len(actual) + eps
    return float(np.sum((a - e) * np.log(a / e)))

psi_vals = {f'feat_{j}': psi(snap1[:, j], snap2[:, j]) for j in range(snap1.shape[1])}
psi_series = pd.Series(psi_vals).sort_values(ascending=False)
print(psi_series.round(4).to_string())
assert psi_series.iloc[0] > 0.25, 'la feature drifteada debe superar el umbral 0.25'
print(f'\ndrift detectado en {psi_series.idxmax()} (PSI={psi_series.max():.3f} > 0.25)')

## 4. KS-test para features continuas

`ks_2samp` compara dos muestras y devuelve un p-valor. p<0.05 ⇒ rechazamos que vengan de la
misma distribución (hay drift).

In [ ]:
stat_drift, p_drift = ks_2samp(snap1[:, feat], snap2[:, feat])
stat_ok, p_ok = ks_2samp(snap1[:, 1], snap2[:, 1])
print(f'feature drifteada  -> KS={stat_drift:.3f}, p={p_drift:.2e}  drift={p_drift < 0.05}')
print(f'feature estable    -> KS={stat_ok:.3f}, p={p_ok:.3f}  drift={p_ok < 0.05}')
assert p_drift < 0.05 <= p_ok, 'KS debe marcar la drifteada y no la estable'

## 5. Shadow deploy + trigger de retraining

Modelo A (viejo) y B (nuevo) reciben los mismos requests; B no responde al usuario, solo se
loguea. Medimos tasa de desacuerdo. Y disparamos alerta si algún PSI supera 0.25.

In [ ]:
model_A = clf
model_B = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=1).fit(Xtr, ytr)
pred_A, pred_B = model_A.predict(snap2), model_B.predict(snap2)
disagree = float(np.mean(pred_A != pred_B))
print(f'shadow deploy - tasa de desacuerdo A vs B: {disagree:.3%}')

RETRAIN = (psi_series > 0.25).any()
print('trigger de retraining:', 'DISPARAR (drift > 0.25)' if RETRAIN else 'no hace falta')
assert RETRAIN, 'con el drift inyectado el trigger deberia dispararse'

## 6. Visual: distribución mes1 vs mes6 y PSI por feature

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(snap1[:, feat], bins=40, alpha=0.6, label='mes 1', color='#37a')
axes[0].hist(snap2[:, feat], bins=40, alpha=0.6, label='mes 6', color='#c33')
axes[0].set_title(f'Drift en feature {feat}'); axes[0].legend()

colors = ['#c33' if v > 0.25 else '#37a' for v in psi_series.values]
axes[1].barh(psi_series.index[::-1], psi_series.values[::-1], color=colors[::-1])
axes[1].axvline(0.25, color='k', ls='--', lw=0.8)
axes[1].set_xlabel('PSI'); axes[1].set_title('PSI por feature (rojo = drift)')
plt.tight_layout(); plt.show()

## Ejercicios

1. Inyectá drift además en la escala (multiplicá una feature por 1.5). ¿El PSI lo detecta o
   solo capta cambios de media? Compará con el KS-test.
2. Simulá *model drift* (no solo data drift): cambiá la relación X→y en el snapshot 6 y medí
   la caída de accuracy con labels reales sobre una ventana móvil.
3. Implementá un trigger por caída de KPI: alertá si el accuracy en holdout móvil baja más de
   3 puntos respecto al baseline.
4. Redactá una Model Card en markdown con: uso previsto, métricas globales y por subgrupo,
   y una sección explícita de *out-of-scope uses*.

## Conclusiones

- El artefacto serializado (`joblib`) es lo que va a producción; debe recargar idéntico.
- Data drift (cambia X) se detecta sin labels con PSI/KS; model drift necesita labels.
- PSI>0.25 o KS con p<0.05 son señales estándar para disparar retraining.
- Shadow deploy valida un modelo nuevo con datos reales y riesgo cero antes del switch.